In [5]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

from langchain_chroma import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings
from dotenv import load_dotenv
from pydantic import BaseModel
from typing import List

In [6]:
# load environment variables from .env file
load_dotenv()

True

In [7]:
# Define the persistent directory for Chroma database
persistent_directory = "db/chroma_db"

# Load embeddings and vector store
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Initialize Chroma vector store with the specified persistent directory and embedding model
db = Chroma(
    persist_directory=persistent_directory,
    embedding_function=embedding_model,
    collection_metadata={"hnsw:space": "cosine"},
)

# load the Google Gemini model for generating responses
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.5)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5869.51it/s]


In [8]:
#Pydantic model for structured output
class QueryVariations(BaseModel):
    queries: List[str]

In [9]:
# ──────────────────────────────────────────────────────────────────
# MAIN EXECUTION
# ──────────────────────────────────────────────────────────────────

# Original query
original_query = "How does Tesla make money?"
print(f"Original Query: {original_query}\n")

Original Query: How does Tesla make money?



In [10]:
# ──────────────────────────────────────────────────────────────────
# Step 1: Generate Multiple Query Variations
# ──────────────────────────────────────────────────────────────────

llm_with_tools = llm.with_structured_output(QueryVariations)

prompt = f"""Generate 3 different variations of this query that would help retrieve relevant documents:

Original query: {original_query}

Return 3 alternative queries that rephrase or approach the same question from different angles."""

response = llm_with_tools.invoke(prompt)
query_variations = response.queries

print("Generated Query Variations:")
for i, variation in enumerate(query_variations, start=1):
    print(f"{i}. {variation}")

print("\n" + "=" * 60)

Generated Query Variations:
1. What are Tesla's primary revenue streams?
2. Describe Tesla's business model and how it generates profit.
3. What are the main sources of income for Tesla Inc.?



In [11]:
# ──────────────────────────────────────────────────────────────────
# Step 2: Search with Each Query Variation & Store Results
# ──────────────────────────────────────────────────────────────────

retriever = db.as_retriever(search_kwargs={"k": 5})  # Get more docs for better RRF
all_retrieval_results = []  # Store all results for RRF

for i, query in enumerate(query_variations, start=1):
    print(f"\n=== RESULTS FOR QUERY {i}: {query} ===")

    docs = retriever.invoke(query)
    all_retrieval_results.append(docs)  # Store for RRF calculation

    print(f"Retrieved {len(docs)} documents:\n")

    for j, doc in enumerate(docs, 1):
        print(f"Document {j}:")
        print(f"{doc.page_content[:150]}...\n")

    print("-" * 50)

print("\n" + "=" * 60)
print("Multi-Query Retrieval Complete!")


=== RESULTS FOR QUERY 1: What are Tesla's primary revenue streams? ===
Retrieved 5 documents:

Document 1:
Finances
For the fiscal (and calendar) year 2021, Tesla
reported a net income of $5.52 billion.[562] The Tesla financial performance

annual revenue w...

Document 2:
60. Root, Al (December 7, 2020). "Tesla Becomes Only the Sixth Company to Top $600 Billion in Value" (https://
www.barrons.com/articles/tesla-becomes-...

Document 3:
Production  1,773,443 vehicles (2024)
A lawsuit settlement agreed to by Eberhard and Tesla in September 2009 output  31.4 GWh battery energy
allows al...

Document 4:
33. Scholer, Kristen; Spears, Lee (June 29, 2010). "Tesla Posts Second-Biggest Rally for 2010 U.S. IPO" (http
s://www.bloomberg.com/news/articles/2010...

Document 5:
568. O'Kane, Sean (July 26, 2021). "Tesla finally made a profit without the help of emission credits" (https://www.t
heverge.com/2021/7/26/22594778/te...

--------------------------------------------------

=== RESULTS FOR Q